# Phase 3 - Step 5: the encoder-decoder's encoder (experiments #4 and #6)

The Phase-1 encoder-decoder (D) has a 4-layer **bidirectional** encoder (~40M parameters with embeddings),
smaller and faster than the 10-layer decoders (~70M). Both experiments use only that encoder, with
1024-token windows (it never read more than 1024 tokens in Phase 1) and one window at a time
(bidirectional attention would see padding).

| #   | Method                                                 | Idea                                                                                                              |
| --- | ------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------- |
| 4   | **Knowledge distillation** (mandatory)                 | the small encoder (student) learns from the gold labels AND from the soft predictions of our best model (teacher) |
| 6   | **rsLoRA** r = 32                                      | LoRA with scale alpha / sqrt(r) instead of alpha / r, so a larger rank still learns at a normal speed             |
| -   | full fine-tuning, no teacher (dev only, not submitted) | control: shows how much KD itself adds                                                                            |

**Teacher = A + LoRA** (`A_lorar8_causal_doc`): best dev macro-F1 (0.584) and best test score so far (0.5735).
Stage A uses the teacher trained on train only (it has never seen the dev labels); Stage B the teacher trained
on train+dev.

**KD loss** per sentence: (1 - a) _ CE(student, gold) + a _ T^2 \* KL(teacher_T || student_T), a = 0.5, **T = 4**.
Phase-2 lesson: the teacher has memorised its training documents, so its plain probabilities are almost one-hot
(no extra information). T = 4 divides the scores by 4 before the softmax, which brings back the teacher's
"second choices" (e.g. 'this Ratio sentence is a bit like PrecedentReliedUpon'). Section 1 measures this.


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, rr_utils, rr_models, rr_experiment
for m in (rr_utils, rr_models, rr_experiment):
    importlib.reload(m)
from rr_experiment import Experiment, compare

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

TEACHER = {"phase1": "A", "method": "lora", "rank": 8, "alpha": 16}
ENC = {"phase1": "D", "max_len": 1024, "stride": 768}
RUNS = {
    "#4 D + KD": {**ENC, "method": "kd", "kd_teacher": TEACHER, "kd_alpha": 0.5, "kd_T": 4.0},
    "#6 D + rsLoRA": {**ENC, "method": "rslora", "rank": 32, "alpha": 16},
    "control: D full, no KD": {**ENC, "method": "full"},
}

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Experiment #4: knowledge distillation into the encoder

The first call computes the teacher's predictions on the 80 training documents once (cached in
`teacher_train.npz`). Below: how confident the teacher is on them, with T = 1 and with T = 4.


In [2]:
exp = Experiment(ROOT, RUNS["#4 D + KD"])
teacher = exp.teacher_outputs("train")
conf = {}
for T in (1.0, 2.0, 4.0):
    p = np.concatenate([np.exp(v / T) / np.exp(v / T).sum(1, keepdims=True) for v in teacher.values()])
    conf[T] = {"mean top probability": round(float(p.max(1).mean()), 3),
               "mean second probability": round(float(np.sort(p, 1)[:, -2].mean()), 3)}
print("teacher on its own training documents:", conf)

experiment: D_kd_bidir_doc_w1024 | device cuda | precision torch.bfloat16
experiment: A_lorar8_causal_doc | device cuda | precision torch.bfloat16


teacher on its own training documents: {1.0: {'mean top probability': 0.941, 'mean second probability': 0.046}, 2.0: {'mean top probability': 0.786, 'mean second probability': 0.095}, 4.0: {'mean top probability': 0.432, 'mean second probability': 0.122}}


In [3]:
exp.stage_a()
exp.report()
exp.stage_b()
exp.speed()
exp.log()

parameters: 42.19M total, 42.191M trainable | loaded {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head']}
[cache] teacher outputs teacher_train.npz


epoch  1 | train loss 5.2151 | 33s | dev macro-F1 0.1611 (CL 0.1638, IT 0.1792) acc 0.3333  <- best


epoch  2 | train loss 3.4085 | 31s | dev macro-F1 0.3087 (CL 0.3089, IT 0.3245) acc 0.4141  <- best


epoch  3 | train loss 2.6214 | 32s | dev macro-F1 0.3310 (CL 0.3328, IT 0.3322) acc 0.4528  <- best


epoch  4 | train loss 2.0092 | 29s | dev macro-F1 0.3991 (CL 0.3393, IT 0.4114) acc 0.4897  <- best


epoch  5 | train loss 1.5363 | 27s | dev macro-F1 0.4276 (CL 0.4196, IT 0.4208) acc 0.5201  <- best


... (23 outputs left out)

epoch 13 | train loss 0.1977 | 25s


epoch 14 | train loss 0.1752 | 25s
training 'full' finished -> loaded best epoch 14


submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\D_kd_bidir_doc_w1024\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 669, 'Fact': 445, 'PrecedentReliedUpon': 342, 'Statute': 170, 'ArgumentPetitioner': 155, 'ArgumentRespondent': 140, 'RulingByLowerCourt': 122, 'PrecedentNotReliedUpon': 51, 'RulingByPresentCourt': 41, 'Dissent': 27, 'None': 24, 'Issue': 11}


speed: {'tokens_per_sec': 171698, 'sentences_per_sec': 3493, 'infer_sec_dev': 0.61, 'peak_gb_infer': 0.25, 'params_total': 42191021, 'params_trainable': 42191021}
logged: D_kd_bidir_doc_w1024 | dev macro-F1 0.4444


## 2. Experiment #6: encoder + rsLoRA (r = 32)


In [4]:
exp = Experiment(ROOT, RUNS["#6 D + rsLoRA"])
exp.stage_a()
exp.report()
exp.stage_b()
exp.speed()
exp.log()

experiment: D_rslorar32_bidir_doc_w1024 | device cuda | precision torch.bfloat16
parameters: 43.59M total, 3.281M trainable | loaded {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head'], 'adapter_params': 1400832}


epoch  1 | train loss 2.3599 | 28s | dev macro-F1 0.1787 (CL 0.1790, IT 0.1952) acc 0.3487  <- best


epoch  2 | train loss 1.7570 | 28s | dev macro-F1 0.3264 (CL 0.2814, IT 0.3319) acc 0.4318  <- best


epoch  3 | train loss 1.5205 | 27s | dev macro-F1 0.3720 (CL 0.3377, IT 0.3852) acc 0.4697  <- best


epoch  4 | train loss 1.3140 | 27s | dev macro-F1 0.3709 (CL 0.3077, IT 0.4308) acc 0.4767


epoch  5 | train loss 1.1266 | 27s | dev macro-F1 0.3941 (CL 0.3476, IT 0.4280) acc 0.4785  <- best


... (17 outputs left out)

epoch  8 | train loss 0.6242 | 49s


epoch  9 | train loss 0.5450 | 46s
training 'full' finished -> loaded best epoch 9


submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\D_rslorar32_bidir_doc_w1024\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 636, 'PrecedentReliedUpon': 408, 'Fact': 390, 'ArgumentRespondent': 197, 'Statute': 170, 'ArgumentPetitioner': 143, 'RulingByLowerCourt': 133, 'RulingByPresentCourt': 39, 'None': 34, 'Issue': 27, 'PrecedentNotReliedUpon': 20}


speed: {'tokens_per_sec': 66559, 'sentences_per_sec': 1354, 'infer_sec_dev': 1.58, 'peak_gb_infer': 0.24, 'params_total': 43591853, 'params_trainable': 3281453}
logged: D_rslorar32_bidir_doc_w1024 | dev macro-F1 0.5299


## 3. Control (dev only): the same student without a teacher

Same encoder, same full fine-tuning, plain cross-entropy. Comparing it with #4 answers
"does distillation help?". Not submitted (it is the same method as #1, only on a different model).


In [5]:
exp = Experiment(ROOT, RUNS["control: D full, no KD"])
exp.stage_a()
exp.report(show_per_label=False)
exp.speed()
exp.log()

experiment: D_full_bidir_doc_w1024 | device cuda | precision torch.bfloat16
parameters: 42.19M total, 42.191M trainable | loaded {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head']}


epoch  1 | train loss 2.2387 | 32s | dev macro-F1 0.2070 (CL 0.2098, IT 0.2149) acc 0.3581  <- best


epoch  2 | train loss 1.5732 | 30s | dev macro-F1 0.3941 (CL 0.3103, IT 0.3943) acc 0.4552  <- best


epoch  3 | train loss 1.2218 | 29s | dev macro-F1 0.3754 (CL 0.3147, IT 0.3795) acc 0.4570


epoch  4 | train loss 0.9678 | 27s | dev macro-F1 0.4282 (CL 0.3469, IT 0.4431) acc 0.4963  <- best


epoch  5 | train loss 0.6868 | 38s | dev macro-F1 0.4353 (CL 0.3764, IT 0.4289) acc 0.4907  <- best


... (2 outputs left out)

epoch  8 | train loss 0.2378 | 30s | dev macro-F1 0.3984 (CL 0.3619, IT 0.3818) acc 0.4795


epoch  9 | train loss 0.1603 | 33s | dev macro-F1 0.4131 (CL 0.3834, IT 0.3826) acc 0.5117
early stop: no improvement for 4 epochs
training 'train' finished -> loaded best epoch 5
Stage A: best epoch 5 | dev macro-F1 0.4353 | 304s | peak GPU 1.09 GB


D_full_bidir_doc_w1024 - dev:
  all: macro-F1 0.4353 | macro-P 0.4608 | macro-R 0.4520 | accuracy 0.4907 | macro-F1 over all 13 0.3684 | 11 labels, 2142 sentences
  CL : macro-F1 0.3764 | macro-P 0.3874 | macro-R 0.4557 | accuracy 0.4893 | macro-F1 over all 13 0.3185 | 11 labels, 1396 sentences
  IT : macro-F1 0.4289 | macro-P 0.4722 | macro-R 0.4179 | accuracy 0.4933 | macro-F1 over all 13 0.3629 | 11 labels, 746 sentences
most confused (gold -> predicted): [('ArgumentPetitioner -> ArgumentRespondent', 173), ('PrecedentReliedUpon -> RatioOfTheDecision', 98), ('ArgumentPetitioner -> RatioOfTheDecision', 67), ('PrecedentReliedUpon -> Fact', 60), ('RatioOfTheDecision -> ArgumentRespondent', 41), ('RatioOfTheDecision -> Fact', 40), ('Fact -> RulingByLowerCourt', 40), ('ArgumentRespondent -> ArgumentPetitioner', 40)]


speed: {'tokens_per_sec': 113866, 'sentences_per_sec': 2316, 'infer_sec_dev': 0.92, 'peak_gb_infer': 0.25, 'params_total': 42191021, 'params_trainable': 42191021}
logged: D_full_bidir_doc_w1024 | dev macro-F1 0.4353


## 4. All Phase-3 runs (dev) + efficiency


In [6]:
compare(ROOT)
for d in sorted((ROOT / "checkpoints" / "phase3").glob("*/speed.json")):
    s = json.load(open(d))
    print(f"{d.parent.name:32s} params {s['params_total'] / 1e6:6.2f}M (trainable {s['params_trainable'] / 1e6:6.2f}M) | "
          f"{s['sentences_per_sec']:5d} sentences/s | inference peak {s['peak_gb_infer']} GB")

run                                     macro     CL     IT    acc  ep     Fact   Issue Argumen Argumen Statute Dissent Precede Precede Precede RulingB RatioOf RulingB    None
A_lorar8_causal_doc                    0.5838 0.4885 0.5939 0.6232  12    0.744   0.710   0.392   0.549   0.812   0.000   0.577   0.394   0.000   0.616   0.731   0.551   0.346
B_lormar8_causal_doc                   0.5727 0.5388 0.5286 0.6433   7    0.753   0.545   0.599   0.560   0.824   0.000   0.634   0.336   0.000   0.450   0.698   0.600   0.300
A2_ia3_causal_doc                      0.5660 0.5159 0.5572 0.6237  12    0.722   0.538   0.488   0.614   0.750   0.000   0.557   0.312   0.000   0.667   0.678   0.592   0.308
A2_full_causal_doc                     0.5616 0.4883 0.5434 0.6228   7    0.715   0.629   0.574   0.561   0.700   0.000   0.530   0.371   0.000   0.529   0.713   0.565   0.292
D_rslorar32_bidir_doc_w1024            0.5299 0.4614 0.5090 0.5924   9    0.748   0.471   0.423   0.533   0.811   0.000 